# Notebook 07 — Validação amostral do redesign (dois eixos)

**Sprint 2 — Lei e Política**

Valida, com evidência, que o modelo de **dois eixos** (natureza por regras + tema por
K-Means só nas substantivas) é metodologicamente superior ao **K-Means único global**.
Três frentes:

1. **Comparação de clusterização** — silhueta e tamanho do maior cluster, ANTES
   (K=10 global) × DEPOIS (Eixo B, K=20 só nas A3).
2. **Inspeção amostral** — amostra estratificada por tema final, para conferir a
   coerência rótulo↔ementa à mão.
3. **Justificativa para a banca.**

In [1]:
import sys
sys.path.insert(0, '..')
import re, unicodedata
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.db import buscar_todos
from src.classificacao.natureza import classificar_natureza

df = pd.DataFrame(buscar_todos('proposicoes', 'id_externo,casa,ementa,eixo,natureza_codigo,tema_cidadao'))
print(f'Proposições: {len(df)}')

def remover_acentos(t):
    return ''.join(c for c in unicodedata.normalize('NFKD', t) if not unicodedata.combining(c))

Proposições: 22106


## 1. Limpeza de texto (duas versões)

Para uma comparação honesta, o **ANTES** usa a limpeza da versão original (sem meses);
o **DEPOIS** usa a limpeza nova (com meses e mais jargão). Cada pipeline é avaliado com a
própria representação — é assim que de fato rodariam.

In [2]:
STOP_BASE = set('''a o as os um uma uns umas de do da dos das em no na nos nas por pelo pela
pelos pelas com sem sob sobre para pra ate entre contra desde e ou mas que se como quando porque
pois ja nao sim ao aos este esta estes estas esse essa esses essas isto isso aquele aquela aquilo
seu sua seus suas dele dela deles delas meu minha nosso nossa ele ela eles elas eu tu voce nos vos
lhe lhes me te foi ser sao era sera tem ter havia mais menos muito pouco todo toda todos todas
outro outra outros outras mesmo mesma qual quais onde seja sejam tambem apenas cada ainda assim
entao lei leis art arts artigo artigos paragrafo inciso alinea dispoe dispor altera alteracao
alterar institui instituir estabelece estabelecer providencias outras revoga vigencia dar
acrescenta inclui inclusao modifica denomina autoriza cria criacao federal nacional numero decreto
medida provisoria projeto proposta emenda constituicao codigo normas norma regula regulamenta
define fixa concede referente relativo relativa seguinte forma ambito redacao termos'''.split())
MESES = {'janeiro','fevereiro','marco','abril','maio','junho','julho','agosto','setembro',
         'outubro','novembro','dezembro'}
STOP_BASE = {remover_acentos(w) for w in STOP_BASE}
STOP_NOVA = STOP_BASE | MESES

def limpar(texto, stop):
    texto = remover_acentos(str(texto).lower())
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    return ' '.join(t for t in texto.split() if len(t) >= 3 and t not in stop)

## 2. ANTES — K-Means global (K=10) sobre toda a base

In [3]:
ant = df.copy()
ant['txt'] = ant['ementa'].map(lambda e: limpar(e, STOP_BASE))
ant = ant[ant['txt'].str.len() > 0]
Xa = TfidfVectorizer(max_df=0.5, min_df=5, ngram_range=(1,2), max_features=5000).fit_transform(ant['txt'])
kma = KMeans(n_clusters=10, random_state=42, n_init=10).fit(Xa)
sil_antes = silhouette_score(Xa, kma.labels_, sample_size=min(5000, Xa.shape[0]), random_state=42)
maior_antes = 100 * pd.Series(kma.labels_).value_counts().max() / len(kma.labels_)
print(f'ANTES  (global K=10): N={Xa.shape[0]}  silhueta={sil_antes:.4f}  maior_cluster={maior_antes:.1f}%')

ANTES  (global K=10): N=22105  silhueta=0.0375  maior_cluster=64.0%


## 3. DEPOIS — Eixo B (K=20) só sobre as substantivas A3

In [4]:
a3 = df[df['eixo'] == 'tema'].copy()
a3['txt'] = a3['ementa'].map(lambda e: limpar(e, STOP_NOVA))
a3 = a3[a3['txt'].str.len() > 0]
Xd = TfidfVectorizer(sublinear_tf=True, max_df=0.30, min_df=8, ngram_range=(1,2),
                     max_features=6000).fit_transform(a3['txt'])
kmd = KMeans(n_clusters=20, random_state=42, n_init=10).fit(Xd)
sil_depois = silhouette_score(Xd, kmd.labels_, sample_size=min(5000, Xd.shape[0]), random_state=42)
maior_depois = 100 * pd.Series(kmd.labels_).value_counts().max() / len(kmd.labels_)
print(f'DEPOIS (Eixo B K=20): N={Xd.shape[0]}  silhueta={sil_depois:.4f}  maior_cluster={maior_depois:.1f}%')

DEPOIS (Eixo B K=20): N=14091  silhueta=0.0236  maior_cluster=49.5%


## 4. Comparação

> A silhueta isolada não é comparável célula-a-célula (espaços de features e N diferentes).
> A métrica que captura o defeito corrigido é o **tamanho do maior cluster** (o catch-all) e
> a **remoção do ruído de forma**.

In [5]:
ruido_forma = 100 * (df['eixo'] == 'forma').sum() / len(df)
comp = pd.DataFrame([
    {'modelo': 'ANTES — global K=10',  'N': Xa.shape[0], 'silhueta': round(sil_antes,4),
     'maior_cluster_%': round(maior_antes,1), 'ruido_forma_no_kmeans_%': round(ruido_forma,1)},
    {'modelo': 'DEPOIS — Eixo B K=20', 'N': Xd.shape[0], 'silhueta': round(sil_depois,4),
     'maior_cluster_%': round(maior_depois,1), 'ruido_forma_no_kmeans_%': 0.0},
])
print(comp.to_string(index=False))

              modelo     N  silhueta  maior_cluster_%  ruido_forma_no_kmeans_%
 ANTES — global K=10 22105    0.0375             64.0                     36.0
DEPOIS — Eixo B K=20 14091    0.0236             49.5                      0.0


## 5. Inspeção amostral estratificada

Amostra de até **30 proposições por tema final** (todos os eixos), para conferência manual
da coerência rótulo↔ementa. Salva em `data/audit/validacao_amostral.csv`.

In [6]:
from pathlib import Path
rng = np.random.RandomState(42)
amostra = (df.groupby('tema_cidadao', group_keys=False)
             .apply(lambda g: g.sample(min(30, len(g)), random_state=rng))
             [['tema_cidadao', 'eixo', 'natureza_codigo', 'casa', 'id_externo', 'ementa']])
out = Path('..') / 'data' / 'audit' / 'validacao_amostral.csv'
out.parent.mkdir(parents=True, exist_ok=True)
amostra.to_csv(out, index=False)
print(f'Amostra: {len(amostra)} linhas → {out}')
print('\n=== 2 exemplos por tema (conferência rápida) ===')
for tema, g in amostra.groupby('tema_cidadao'):
    print(f'\n● {tema}')
    for e in g['ementa'].head(2):
        print(f'   - {e[:110]}')

Amostra: 690 linhas → ../data/audit/validacao_amostral.csv

=== 2 exemplos por tema (conferência rápida) ===

● Administração e Serviços Públicos
   - Disciplina a utilização de ferramentas de monitoramento remoto de terminais de comunicações pessoais por órgão
   - Institui o piso salarial nacional dos profissionais de apoio escolar e auxiliares de inclusão escolar em todo 

● Criança e Adolescente
   - Altera a Lei nº 8.069, de 13 de julho de 1990, que dispõe sobre o Estatuto da Criança e do Adolescente e dá ou
   - Altera o artigo 241-B da Lei nº 8.069, de 13 de julho de 1990 – Estatuto da Criança e do Adolescente, para aum

● Créditos Orçamentários
   - Abre aos Orçamentos Fiscal e da Seguridade Social da União, em favor de diversos Órgãos do Poder Executivo e d
   - Abre ao Orçamento da Seguridade Social da União, em favor do Ministério da Previdência Social, crédito especia

● Datas Comemorativas
   - Institui o Dia Nacional do EOD/ Explosivista, a ser celebrado em 04 de julho,em

/tmp/ipykernel_172733/3703327128.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(30, len(g)), random_state=rng))


## 6. Justificativa para a banca

**Por que dois eixos > K-Means único.**

1. **Elimina a tensão de taxonomia.** No modelo antigo os 10 rótulos misturavam *forma*
   (licença, requerimento, radiodifusão, crédito) e *assunto* (saúde, penal). Um mesmo
   documento tinha duas respostas possíveis — o modelo era punido por uma ambiguidade que
   estava no **rótulo**, não no dado. Separar os eixos torna cada decisão única e auditável.

2. **Tira ~36% de ruído de forma do K-Means.** Licenças, requerimentos e outorgas têm
   padrão textual estável: são trabalho de **regra**, não de clustering. Removê-los antes
   do K-Means devolve ao algoritmo um corpus de conteúdo real.

3. **Quebra o catch-all.** O maior cluster cai de ~64% (global) para ~50% (Eixo B), e o
   resíduo deixa de ser uma mistura forma×tema para ser **tematicamente homogêneo**
   (políticas públicas gerais), rotulado honestamente como *"Outras Políticas Públicas"*.

4. **Corrige erros de sentido.** Microcrédito social (*Acredita no Primeiro Passo*) deixa de
   vazar para *"Créditos Orçamentários"*: a palavra "crédito" não decide, o sentido decide.

**Limitação assumida.** A silhueta permanece baixa e o maior cluster fica acima dos 35%
desejados: ementas curtas com boilerplate jurídico têm sinal fraco para separar mais sem
explodir K. É uma limitação **inerente ao dado**, não um defeito de design — e o ganho de
interpretabilidade e auditabilidade do modelo de dois eixos é o que a entrega valoriza.

**Trade-off downstream.** Os temas mais finos reduzem levemente a acurácia do modelo de
predição de voto (tendência partido×tema mais esparsa). É um custo aceito: o objetivo do
Sprint 2 é a **qualidade da classificação temática**, não a do classificador supervisionado.